# RAM Graph Energy — piloto de 36 runs

Un seed (42), ambos datasets, tres arquitecturas, seis métodos y cinco épocas. El piloto mide tiempos y memoria; **no** es el benchmark principal. No consulta el conjunto de test.


## 1. Instalar el paquete


In [ ]:
import os, pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/abelalv/energia-2.git"
REPO_DIR = pathlib.Path("/kaggle/working/energia-2")

if not (REPO_DIR / "pyproject.toml").exists():
    candidates = [
        path
        for path in pathlib.Path("/kaggle/input").rglob("pyproject.toml")
        if (path.parent / "src" / "ram_experiments").exists()
    ]
    if candidates:
        shutil.copytree(candidates[0].parent, REPO_DIR, dirs_exist_ok=True)
        print("repo copiado desde", candidates[0].parent)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
# Los subprocesos del notebook deben encontrar el paquete también sin instalación editable.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR / "src"), os.environ.get("PYTHONPATH", "")]
)
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
        check=True,
    )
except subprocess.CalledProcessError:
    # Kaggle sin red o sin uv_build: usar src/ directamente. El entorno sigue siendo el de Kaggle.
    sys.path.insert(0, str(REPO_DIR / "src"))
    print("pip install -e falló; usando src/ en sys.path")
print("cwd:", pathlib.Path.cwd())


## 2. Entorno


In [ ]:
import platform, torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("vram GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# Protocol RAM-2026-10-13-v1: float32 without AMP/TF32.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("tf32:", torch.backends.cuda.matmul.allow_tf32)


## 3. Datos (descarga o dataset adjunto)


In [ ]:
import os, pathlib, subprocess, sys

DATA_DIR = pathlib.Path(os.environ.get("RAM_DATA_DIR", "/kaggle/working/data/processed"))

if not (DATA_DIR / "mnist_main.npz").exists():
    attached = sorted(pathlib.Path("/kaggle/input").rglob("mnist_main.npz"))
    if attached:
        DATA_DIR = attached[0].parent
        print("datos adjuntos:", DATA_DIR)
    else:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "ram_experiments.cli",
                "prepare-data",
                "--datasets",
                "mnist,fashion_mnist",
                "--raw-dir",
                "/kaggle/working/data/raw",
                "--processed-dir",
                str(DATA_DIR),
            ],
            check=True,
        )

print("data dir:", DATA_DIR)
print(sorted(path.name for path in DATA_DIR.glob("*.npz")))
print("define RAM_DATA_DIR en las celdas de ejecución:", DATA_DIR)


## 4. Manifiesto, inventario y ejecución

`MAX_SECONDS=30000` deja margen dentro de la sesión GPU de Kaggle. Si se agota, el runner guarda checkpoint y la próxima sesión reanuda.


In [ ]:
import pathlib, subprocess, sys

MANIFEST = pathlib.Path("configs/pilot_manifest.jsonl")
RUN_ROOT = "/kaggle/working/runs"
MAX_SECONDS = 30_000  # 8 h 20 min; deja margen para guardar la sesión

subprocess.run(
    [
        sys.executable,
        "-m",
        "ram_experiments.cli",
        "manifest",
        "--stage",
        "pilot",
        "--out",
        str(MANIFEST),
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "ram_experiments.cli", "inventory", "--manifest", str(MANIFEST)],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "ram_experiments.cli",
        "run",
        "--manifest",
        str(MANIFEST),
        "--stage",
        "pilot",
        "--run-root",
        RUN_ROOT,
        "--data-dir",
        str(DATA_DIR),
        "--device",
        "cuda",
        "--max-seconds",
        str(MAX_SECONDS),
        "--resume",
    ],
    check=True,
)


## 5. Tabla de tiempos y métricas


In [ ]:
import pathlib, subprocess, sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "ram_experiments.cli",
        "analyze",
        "--manifest",
        "configs/pilot_manifest.jsonl",
        "--run-root",
        "/kaggle/working/runs",
        "--out",
        "/kaggle/working/analysis/pilot",
    ],
    check=True,
)


## 6. Qué devolver a los autores

Tiempo por método/arquitectura/dataset, memoria pico, coste de evaluación, checkpoints y E4. El benchmark principal **no** se lanza hasta acordar el presupuesto medido.
